In [32]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt

In [33]:
with open('generated_data/security_data_generated.json') as file:
    json_data = json.load(file)

events = pd.DataFrame(json_data['events'])
entities = pd.DataFrame(json_data['entities'])
relationships = pd.DataFrame(json_data['relationships'])

attack_scenarios = pd.DataFrame(json_data['ground_truth']['attack_scenarios'])

In [34]:
events

,id,type,timestamp,severity,description,alert_style,alert_message
0,63c20286-3725-403f-a56f-23e1de5697a8,Data Access,2025-06-11T22:31:01.874111,3,Bulk file download from file share,raw_telemetry,DATA ACCESS: jsmith downloaded 3718 files from...
1,9bcae546-b49d-462a-8695-8ecdf184d37c,Defense Evasion,2025-06-07T11:06:29.221649,10,Logs cleared or deleted,raw_telemetry,POTENTIAL DEFENSE EVASION: Logs cleared or del...
2,626bfb38-a2e3-48dc-bf43-166a7ceb0b92,Defense Evasion,2025-06-08T13:37:23.756920,3,AV/EDR process terminated,raw_telemetry,DEFENSE EVASION: Security tool mshta.exe kille...
3,dec26172-db80-4557-ac73-efe527e66443,Authentication,2025-06-11T16:03:11.156486,6,Service account used interactively,raw_telemetry,Windows Event ID 4624: Service account used in...
4,7b9eb301-aaec-4d7c-a9fb-c0bd0ad0c47b,Authentication,2025-06-11T12:24:54.713571,1,Brute-force login attempt,behavioral,ALERT: Multiple failed logins for jsmith from ...
...,...,...,...,...,...,...,...
928,7fece6ec-d7bb-4cbf-bab3-c95f5502843f,Data Access,2025-06-05T15:01:01.440077,8,Bulk file download from file share,rule_based,DATA ACCESS: blee downloaded 3850 files from \...
929,0eb63829-e7ab-49e3-972c-a1788e8d8eef,Data Access,2025-06-05T16:14:53.136497,8,Bulk file download from file share,rule_based,DATA ACCESS: adavis downloaded 1107 files from...
930,1710220c-7c1b-43ef-8674-429b441b775d,Exfiltration,2025-06-05T18:11:39.481171,7,Large data transfer to external IP,rule_based,EXFIL ALERT: 4780 MB sent from 10.51.245.248 t...
931,7925fe1b-3f69-4c6a-9cc3-e06ded69136a,Defense Evasion,2025-06-05T17:17:46.327085,7,Logs cleared or deleted,raw_telemetry,POTENTIAL DEFENSE EVASION: Logs cleared or del...


In [35]:
entities

,id,type,name,properties
0,85f3660d-ae3a-4757-8959-c04f0e341c4a,Host,WS-000,"{'hostname': 'WS-000', 'ip_address': '10.12.14..."
1,e6314556-586a-4a1c-8744-95bd0841b267,Host,WS-001,"{'hostname': 'WS-001', 'ip_address': '10.44.21..."
2,a928edb3-f2df-454a-8fbf-c2b5cf20346b,Host,DC-002,"{'hostname': 'DC-002', 'ip_address': '10.119.1..."
3,a9a6cbf5-11a2-468c-8ed5-2e60834e136d,Host,LT-003,"{'hostname': 'LT-003', 'ip_address': '10.112.2..."
4,c3aa83ab-b0c6-48d8-821f-b5cc28a473b8,Host,WS-004,"{'hostname': 'WS-004', 'ip_address': '10.81.21..."
...,...,...,...,...
483,350f42b5-fecb-4eb2-800d-13a28d655e76,Database,logs_db,"{'name': 'logs_db', 'type': 'PostgreSQL', 'siz..."
484,ad68eea9-23b4-4d01-a4b6-f4139675136f,Database,hr_db,"{'name': 'hr_db', 'type': 'PostgreSQL', 'size'..."
485,cbf75793-5fca-4f52-b25f-4f1a7cdb2bd7,Database,hr_db,"{'name': 'hr_db', 'type': 'SQL Server', 'size'..."
486,f01ec073-5cd3-4b68-be97-d6b7c50b05c3,Database,inventory_db,"{'name': 'inventory_db', 'type': 'MySQL', 'siz..."


In [36]:
relationships

,source,target,weight,type
0,63c20286-3725-403f-a56f-23e1de5697a8,68a2a098-f33f-48aa-9361-13649408dcd6,1.0,related_to
1,63c20286-3725-403f-a56f-23e1de5697a8,561e4570-d9b2-444f-b16b-b6e6000e82dd,1.0,related_to
2,63c20286-3725-403f-a56f-23e1de5697a8,cbe03784-710a-4572-a9b0-5a76e5159ace,1.0,related_to
3,63c20286-3725-403f-a56f-23e1de5697a8,af31bbe7-8957-4d3f-8bdd-bf8980e6dcad,1.0,related_to
4,9bcae546-b49d-462a-8695-8ecdf184d37c,89c08c6c-c380-4372-a3dd-32f04218c500,1.0,related_to
...,...,...,...,...
3237,7925fe1b-3f69-4c6a-9cc3-e06ded69136a,46a604cf-d935-42e8-8333-a376c14aa3c5,1.0,related_to
3238,7925fe1b-3f69-4c6a-9cc3-e06ded69136a,929e07ee-4aa0-4f4a-9812-e7af46fafe59,1.0,related_to
3239,a313b8d6-77b5-45dc-8d28-adda5c8e4d7a,e79a6aa0-a563-40c0-9ac8-148b459b2c6c,1.0,related_to
3240,a313b8d6-77b5-45dc-8d28-adda5c8e4d7a,7068b181-d557-454f-845f-b19253dd0114,1.0,related_to


In [37]:
attack_scenarios

,id,description,event_ids,entity_ids
0,cf8f624e-51f4-4c3f-9b57-ba12cf2cc70a,Attack Scenario: Ransomware,"[904e2cb8-3a45-4560-b9b7-0ddcb919ac89, f9e3ffc...","[c114ec03-2c7a-44c6-b481-6612387c8277, e631455..."
1,975faae4-9317-4fd2-b071-2c209a59efc9,Attack Scenario: Supply Chain Attack,"[1f08951a-72c3-422e-8212-32756cfadefb, c3e5268...","[111e31d3-5626-4f41-ad5f-ddbb3bde738a, b54379a..."
2,879d6473-a957-433c-aacc-c9d91cabc173,Attack Scenario: Insider Threat,"[99aec5d4-59b8-45d0-a1b3-e05527e3d68d, b6b632a...","[535b3acc-6844-4bb9-9280-6cb9a8fd05ae, 6b0dd09..."
3,5b65bac6-6bf1-49b4-86ab-2d5cd30cb0b3,Attack Scenario: Credential Stuffing,"[27c31b6c-c8f3-4893-a131-2076979ca31f, 474e32d...","[92f3a4bd-1527-40c4-9827-a1cc5a1fcf64, cc16603..."
4,43b5636a-7aee-4932-9bad-8841e5732d3a,Attack Scenario: Lateral Movement Campaign,"[f08a3a9d-32e0-4f0b-8acc-59ad0e1699ef, 602d680...","[b54379a8-ccc1-4b16-bae1-5089adc7e977, 335f7fe..."
5,683da449-e5da-4301-99e8-e9c2d1271bed,Attack Scenario: Data Exfiltration,"[7fece6ec-d7bb-4cbf-bab3-c95f5502843f, 0eb6382...","[300da682-c6c4-4794-818d-29647820b817, 6b0dd09..."


In [38]:
'''
Sanity Check
====================================================================================
The event_ids that appear in the ground_truth.attack_scenarios array are precisely
those that are listed in ground_truth.true_positive_events.
'''

e = [event for i in range(len(attack_scenarios)) for event in attack_scenarios['event_ids'][i]]
print(set(e) == set(json_data['ground_truth']['true_positive_events'])) # should print true

'''
Likewise, the entity_ids that appear in the ground_truth.attack_scenarios array are precisely
those that are listed in ground_truth.true_positive_entities.
'''

e = [entity for i in range(len(attack_scenarios)) for entity in attack_scenarios['entity_ids'][i]]
print(set(e) == set(json_data['ground_truth']['true_positive_entities'])) # should print true

KeyError: 'true_positive_events'

In [8]:
def create_bipartite_graph(part1, part2, edges) -> nx.Graph:
    """Load json file and return a bipartite weighted graph.

    Nodes are the `id` values from `events` and `entities`.
    Edges are created from `relationships` with the edge attribute `weight` set
    from the relationship's `weight` field. Each node gets attributes:
    - `kind`: either 'event' or 'entity'
    - `bipartite`: 0 for events, 1 for entities
    """
    G = nx.Graph()

    part1_ids = part1["id"]
    part2_ids = part2["id"]

    for nid in part1_ids:
        G.add_node(nid, bipartite=0)
    for nid in part2_ids:
        G.add_node(nid, bipartite=1)

    for i, e in edges.iterrows():
        src = e["source"]
        tgt = e["target"]
        weight = e["weight"]
        rtype = e["type"]

        if src is None or tgt is None:
            continue

        # Ensure nodes exist in graph (in case relationships reference missing ids)
        if src not in G:
            print("source node not in G")
            G.add_node(src, kind=("event" if src in event_ids else "entity"), bipartite=(0 if src in event_ids else 1))

        if tgt not in G:
            print("target node not in G")
            G.add_node(tgt, kind=("event" if tgt in event_ids else "entity"), bipartite=(0 if tgt in event_ids else 1))

        G.add_edge(src, tgt, weight=weight, type=rtype)

    return G

def connected_components_sorted(G: nx.Graph):
    """Return connected components sorted by size (descending)."""
    comps = list(nx.connected_components(G))
    comps.sort(key=len, reverse=True)
    return comps

In [39]:
G = create_bipartite_graph(events, entities, relationships)
print(f"Loaded graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

from networkx.algorithms import bipartite

#print("Is bipartite ", bipartite.is_bipartite(G))
	# show a few sample edges
#for u, v, d in list(G.edges(data=True))[:1000]:
#	print(u, "-", v, ":", d)

comps = connected_components_sorted(G)
print(f"Connected components: {len(comps)}")
# print sizes of the 5 largest components
for i, comp in enumerate(comps[1:2], start=1):
    #print(f"Component {i}: size={len(comp)}")
    plt.figure(i)
    print("asdfsadf", bipartite.is_bipartite(nx.induced_subgraph(G, comp)))
    nx.draw_bipartite(nx.induced_subgraph(G, comp), with_labels=True)

plt.show()
#This is a test change.

Loaded graph: 1421 nodes, 3242 edges
Connected components: 1
